In [ ]:
import os
import sys

sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.dirname(os.getcwd()))
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "DeepUnitMatch"))

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
import umap.umap_ as umap
from tqdm import tqdm
import h5py

from DeepUnitMatch.utils.umap_utils import embed_data



In [ ]:
# Forward pass the data through the network

umap_path = r"/path/to/save/UMAP/data"
data_root = r"/path/to/full/dataset"
save_path = r"/path/where/you/saved/waveform/snippets"
all_mice = [
    "AL031",
    "AL032",
    "AL036",
    "AV008",
    "CB015",
    "CB016",
    "CB017",
    "CB018",
    "CB020",
    "EB019",
    "AV009",
    "AV015",
    "AV021",
    "AV049",
    "EB014",
    "FT033",
    "FT039",
    "JF084",
]
embedded_1, embedded_2, mouse, bc, depths = embed_data(
    data_root, save_path, all_mice, load_bombcell=True, load_depths=True
)

np.save(os.path.join(umap_path, "first_half.npy"), embedded_1)
np.save(os.path.join(umap_path, "second_half.npy"), embedded_2)
pd.DataFrame(bc).to_csv(os.path.join(umap_path, "bombcell.csv"))
np.save(os.path.join(umap_path, "depths.npy"), np.array(depths))

In [ ]:
# Compute and save UMAP embeddings

n_neighbours = 5
min_dist = 0.1
n_components = 2
metric = "euclidean"

umap_embedding = umap.UMAP(n_neighbours, n_components, metric, min_dist=min_dist, random_state=0).fit_transform(0.5 * (embedded_1 + embedded_2), axis=0)
np.save(os.path.join(umap_path, "UMAPembeddings.npy"), umap_embedding)

In [ ]:
# Plot the UMAP embeddings
single_mouse = None

umap_embedding = np.load(os.path.join(umap_path, "UMAPembeddings.npy"))

fig, ax = plt.subplots(figsize=(11, 8), constrained_layout=False)
if single_mouse:
    labels = [
        all_mice[index]
        for index in mouse
        if all_mice[index] == single_mouse
    ]
    indices = [i for i in mouse if all_mice[i] == single_mouse]
    umap_to_plot = np.array(
        [
            umap_embedding[i, :]
            for i in range(len(mouse))
            if all_mice[mouse[i]] == single_mouse
        ]
    )
else:
    labels = [all_mice[index] for index in mouse]
    indices = mouse
    umap_to_plot = umap_embedding

scatter = ax.scatter(
    x=umap_to_plot[:, 0], y=umap_to_plot[:, 1], s=0.3, c=indices, label=labels
)
unique_labels = dict(zip(indices, labels))  # Remove duplicates
handles = [
    plt.Line2D(
        [],
        [],
        marker="o",
        linestyle="",
        markersize=5,
        color=scatter.cmap(scatter.norm(m)),
    )
    for m in unique_labels.keys()
]

ax.legend(handles, unique_labels.values())

plt.show()

In [ ]:
# Colour code by depth

single_mouse = "AV008"

depths = np.array(depths) / 1000

AV8depths = []
for i, m in enumerate(mouse):
    if m == 3:
        AV8depths.append(depths[i])

fig, ax = plt.subplots(figsize=(11, 8), constrained_layout=False)
if single_mouse:
    labels = [
        all_mice[index]
        for index in mouse
        if all_mice[index] == single_mouse
    ]
    indices = [i for i in mouse if all_mice[i] == single_mouse]
    umap_to_plot = np.array(
        [
            umap_embedding[i, :]
            for i in range(len(mouse))
            if all_mice[mouse[i]] == single_mouse
        ]
    )
else:
    labels = [all_mice[index] for index in mouse]
    indices = mouse
    umap_to_plot = umap_embedding

scatter = ax.scatter(
    x=umap_to_plot[:, 0],
    y=umap_to_plot[:, 1],
    s=0.3,
    c=AV8depths,
    cmap="viridis",
    label=labels,
)
cbar = plt.colorbar(scatter, ax=ax, pad=0.01, label="Depth (mm)")

In [ ]:
# Colour code UMAP by Bombcell output
bc_param = "waveformDuration_peakTrough"
# bc_param = "spatialDecaySlope"

umap_to_plot = umap_embedding

if bc_param == "rawAmplitude":
    df = pd.DataFrame(bc)
    df = df.loc[df["rawAmplitude"] < 500]
    mask = np.zeros(len(bc[bc_param]), dtype=bool)
    for i in df.index:
        mask[i] = True
    umap_to_plot = umap_embedding[mask]
    colours = df[bc_param].values
elif bc_param == "nSpikes":
    df = pd.DataFrame(bc)
    df = df.loc[df["nSpikes"] < 50000]
    mask = np.zeros(len(bc[bc_param]), dtype=bool)
    for i in df.index:
        mask[i] = True
    umap_to_plot = umap_embedding[mask]
    colours = df[bc_param].values
elif bc_param == "spatialDecaySlope":
    df = pd.DataFrame(bc)
    df = df.loc[df["spatialDecaySlope"] < 0]
    df = df.loc[df["spatialDecaySlope"] > -0.015]
    mask = np.zeros(len(bc[bc_param]), dtype=bool)
    for i in df.index:
        mask[i] = True
    umap_to_plot = umap_embedding[mask]
    colours = df[bc_param].values
else:
    umap_to_plot = umap_embedding
    colours = bc[bc_param]

fig, ax = plt.subplots(figsize=(11, 8), constrained_layout=False)
cmap = plt.cm.viridis
scatter = ax.scatter(
    x=umap_to_plot[:, 0],
    y=umap_to_plot[:, 1],
    s=0.3,  # Small point size
    c=colours,
    cmap=cmap,
    alpha=0.8,
)
cbar = plt.colorbar(scatter, ax=ax, pad=0.01)
cbar.set_label("Waveform Duration", fontsize=14)
ax.set_title("UMAP Visualisation Colored by Waveform Duration", fontsize=14)
ax.set_xlabel("UMAP Dimension 1", fontsize=14)
ax.set_ylabel("UMAP Dimension 2", fontsize=14)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.rcParams["svg.fonttype"] = "none"
ax = plt.gca()
ax.spines[["right", "top"]].set_visible(False)
plt.show()